<br>
<p float="right">
  <img src=attachment:nanoHUB_logo_color.png width="25%" height='10%' align="right" /> 
</p>

# 2. Normalizer and Parsing

### <i>Ethan Holbrook, Juan C. Verduzco, </i>  and <i>Alejandro Strachan </i>
### Materials Engineering, Purdue University <br>

## Overview

This notebook is the second stage of the evaluation pipeline. It reads generated LAMMPS scripts, applies the `lammps_ast` sanitizer, and attempts to parse each sanitized script into an abstract syntax tree (AST).

The resulting parse outcomes provide a structural validity check before execution. Sanitized scripts, AST objects, and summary parsing results produced here are used as inputs to the execution and accuracy stages that follow.


## Tips
1. Found a bug? Email holbrooe@purdue.edu
2. Sanitizer and Normalizer are interchangeable words describing the same thing. 

<br><br><br>

# Libraries

In [46]:
import os
import json
import sys
import shutil
import re
import pickle

from dotenv import load_dotenv
load_dotenv()

from lark import tree
from lammps_ast.sanitizer import sanitize
from lammps_ast.parser import parse_to_AST


###################################

import os
from lark import Lark
from colorama import Fore, Style
from lammps_ast.transformer import RemoveNewlines

#####################

from dataclasses import dataclass

@dataclass
class ParseErrorInfo:
    line: int
    column: int
    token: str
    text: str

GRAMMAR_PATH =  "../lammps_ast/grammar/lammps_grammar.lark" ## your path to grammar
with open(GRAMMAR_PATH, "r") as f:
    LAMMPS_GRAMMAR = f.read()

# Initialize the parser using the built-in grammar
parser = Lark(LAMMPS_GRAMMAR, parser="lalr", keep_all_tokens=True)

def parse_to_AST(sanitized_script, *, lint=False, max_errors=10, verbose=False):
    """
    If lint=False (default):
        returns (parse_tree, None) on success
        returns (None, err) on failure

    If lint=True:
        returns (parse_tree_or_None, [ParseErrorInfo, ...])
        - parse_tree_or_None is a valid tree only if parsing eventually succeeds
        - errors contains up to max_errors items
    """
    # --- Parser-only mode (your current behavior, but optionally quiet) ---
    if not lint:
        try:
            parse_tree = parser.parse(sanitized_script)
            parse_tree = RemoveNewlines().transform(parse_tree)
            return parse_tree, None
        except Exception as e:
            if verbose:
                print(f""" \t {Fore.RED}🟥 Critical Parse Error:{Style.RESET_ALL}.
                    Unexpected token {repr(e.token)} at line {e.line}, column {e.column}.
                    Expected one of: {e.expected}.
                    Previous token: {e.token_history}""")
            return None, e

    # --- Linter mode (collect multiple errors) ---
    lines = sanitized_script.splitlines(True)  # preserve newlines
    errors = []

    for _ in range(max_errors):
        try:
            parse_tree = parser.parse("".join(lines))
            parse_tree = RemoveNewlines().transform(parse_tree)
            return parse_tree, errors  # success with collected errors (possibly empty)
        except Exception as e:
            line_idx = e.line - 1
            bad_line = lines[line_idx].rstrip("\n") if 0 <= line_idx < len(lines) else ""

            err_info = ParseErrorInfo(
                line=getattr(e, "line", None),
                column=getattr(e, "column", None),
                token=repr(getattr(e, "token", None)),
                text=bad_line
            )
            errors.append(err_info)

            if verbose:
                print(f""" \t {Fore.RED}🟥 Parse Error:{Style.RESET_ALL}.
                    Unexpected token {repr(getattr(e,'token',None))} at line {getattr(e,'line',None)}, column {getattr(e,'column',None)}.
                    Expected one of: {getattr(e,'expected',None)}.
                    Previous token: {getattr(e,'token_history',None)}
                    Line: {bad_line}""")

            # Prevent infinite loops / out-of-range
            if not (0 <= line_idx < len(lines)):
                break

            # Neutralize the offending line but keep line numbering
            newline = "\n" if lines[line_idx].endswith("\n") else ""
            lines[line_idx] = "\n" if lines[line_idx].endswith("\n") else ""

            # If we keep hitting the same spot, stop
            if len(errors) >= 2 and errors[-1].line == errors[-2].line and errors[-1].column == errors[-2].column:
                break

    return None, errors


###################################
    

import lammps_ast
print(dir(lammps_ast))
print(lammps_ast.__path__)

import openai
from openai import OpenAI

import numpy as np
import pandas as pd

import subprocess

from colorama import Fore, Style

from importlib.metadata import version
print(version("lammps-ast"))

# import anthropic

['__author__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'error_handler', 'grammar', 'parse_to_AST', 'parser', 'sanitize', 'sanitizer', 'transformer']
['/home/holbrooe/.conda/envs/2022.10-py39/gst/lib/python3.12/site-packages/lammps_ast']
0.1.7


## Initialization of directory variables

In [29]:
current_dir = os.getcwd()
# parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
parent_dir = os.path.abspath(os.path.join(current_dir))
scripts_dir = os.path.join(parent_dir,"generated_scripts")
scripts_base = os.path.join(parent_dir,"generated_scripts")

print(scripts_dir)

prompt_dirs = ['prompt1','prompt2','prompt3']
model_dirs = ['gpt-4o','gpt-4.1','gpt-o3','claude-opus-4','gpt-5'] #,'gpt-4o-search',]
# model_dirs = [] #,'gpt-4o-search',]
model_names = ['gpt-4o-2024-08-06','gpt-4.1-2025-04-14','o3-2025-04-16','claude-4-opus-20250514','gpt-5-2025-08-07']
# model_names = ['gpt-5-2025-08-07']

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts


In [30]:
prompt_choice = 1
model_choice = 0

prompt_type = prompt_dirs[prompt_choice-1]
print(prompt_type)
model_type = model_dirs[model_choice]
model_name = model_names[model_choice]
print(model_type,model_name)

trial=0
# output_filename = os.path.join(prompt_type, model_type,f'P_{prompt_type}-M_{model_type}-T_{trial}.in')

print(prompt_dirs)
print(model_dirs)
print(model_names)

# prompt_dict = {'prompt1':prompts[0],'prompt2':prompts[1],'prompt3':prompts[2]}

model_dict = {model_dirs[0]:model_names[0],
              model_dirs[1]:model_names[1],
              model_dirs[2]:model_names[2],
              model_dirs[3]:model_names[3],
              model_dirs[4]:model_names[4],
             }

prompt1
gpt-4o gpt-4o-2024-08-06
['prompt1', 'prompt2', 'prompt3']
['gpt-4o', 'gpt-4.1', 'gpt-o3', 'claude-opus-4', 'gpt-5']
['gpt-4o-2024-08-06', 'gpt-4.1-2025-04-14', 'o3-2025-04-16', 'claude-4-opus-20250514', 'gpt-5-2025-08-07']


In [ ]:

import re
import math
import simpleeval

def remove_comments(script):
    """Removes inline comments while preserving meaningful lines."""
    return '\n'.join(line.split('#', 1)[0].rstrip() for line in script.splitlines() if line.split('#', 1)[0].strip())

def remove_prints(script):
    """Removes lines that start with 'print'."""
    return '\n'.join(line for line in script.splitlines() if not line.lstrip().startswith('print'))


def merge_ampersand_lines(script):
    """Merges lines ending with '&' into a single line while preserving spacing."""
    merged_lines, buffer = [], None

    for line in script.splitlines():
        stripped = line.rstrip()
        if stripped.endswith('&'):
            buffer = (buffer or "") + " " + stripped[:-1].strip()
        else:
            merged_lines.append((buffer + " " + stripped).strip() if buffer else stripped)
            buffer = None  # Reset buffer after appending

    if buffer:
        merged_lines.append(buffer.strip())

    return '\n'.join(merged_lines)

def parse_variable_line(line):
    """Extracts variable name and expression from a LAMMPS variable definition line."""
    # tokens = line.split(maxsplit=3)  # Adjusted to ensure proper parsing
    tokens = line.split()  # Adjusted to ensure proper parsing

    if len(tokens) < 3 or tokens[0] != "variable":
        return None, None

    var_name = tokens[1]
    var_type = tokens[2]

    # Handle "index" type variables correctly
    if var_type == "index":
        return var_name, tokens[3] if len(tokens) == 4 else None

    # Handle "equal" type variables
    if var_type == "equal": # should return var_name, None for too many spaces issue
        return var_name, tokens[3] if len(tokens) == 4 else None

    return None, None

def process_and_evaluate_variables(script):
    """Replaces variables (`${var}` and `v_var`) while ensuring dependencies are handled iteratively."""
    script_lines = script.splitlines()
    var_dict, variable_definitions, processed_lines = {}, {}, []

    # Extract variable definitions
    for line in script_lines:
        if line.startswith('variable'):
            var_name, expr = parse_variable_line(line)
            if var_name and not expr:
                print('''---------------------
                ---------------------------
                ---------------------------''')
                print(script, var_name)
                return '\n'.join(script_lines)
            if var_name and expr:
                variable_definitions[var_name] = expr
        else:
            processed_lines.append(line)

    # Extract dependencies between variables
    dependency_graph = {}
    for var, expr in variable_definitions.items():
        dependencies = set(re.findall(r'v_([a-zA-Z_]\w*)|\${([a-zA-Z_]\w*)}', expr))
        dependency_graph[var] = {v[0] or v[1] for v in dependencies if (v[0] or v[1]) in variable_definitions}

    # Resolve variables in topological order
    resolved_vars = set()
    while variable_definitions:
        progress_made = False
        for var_name, expr in list(variable_definitions.items()):
            # Only evaluate if all dependencies are resolved
            if dependency_graph[var_name].issubset(resolved_vars):
                expr = re.sub(r'v_([a-zA-Z_]\w*)|\${([a-zA-Z_]\w*)}', lambda m: str(var_dict.get(m.group(1) or m.group(2), f'v_{m.group(1) or m.group(2)}')), expr)
                expr = expr.replace('^', '**').replace('sqrt(', 'math.sqrt(')

                try:
                    result = simpleeval.simple_eval(expr, names={"pi": math.pi}, functions={"sqrt": math.sqrt,"ceil": math.ceil,"floor":math.floor,'exp':math.exp})
                    if isinstance(result, float) and result.is_integer():
                        result = int(result)
                    var_dict[var_name] = str(result)
                    resolved_vars.add(var_name)
                    del variable_definitions[var_name]
                    progress_made = True
                except (NameError, SyntaxError, TypeError) as e:
                    continue  # Skip if an undefined variable is encountered

        if not progress_made:
            break  # Prevent infinite loops

    # Replace variables in the script
    new_lines = []
    for line in processed_lines:
        line = re.sub(r'v_([a-zA-Z_]\w*)|\${([a-zA-Z_]\w*)}', lambda m: str(var_dict.get(m.group(1) or m.group(2), f'v_{m.group(1) or m.group(2)}')), line)
        new_lines.append(line)

    return '\n'.join(new_lines)

def evaluate_expressions(script):
    """Evaluates only pure numeric expressions in the script."""
    arithmetic_pattern = re.compile(r'^[\d+\-*/().eE]+$')  # Optimized regex

    def evaluate_token(token):
        if arithmetic_pattern.fullmatch(token):  # Check if the token is a pure expression
            try:
                value = simpleeval.simple_eval(token.replace('^', '**'), names={"pi": math.pi}, functions={"sqrt": math.sqrt})
                if isinstance(value, float) and value.is_integer():
                    value = int(value)
                return str(value)
            except:
                pass  # Leave token unchanged if evaluation fails
        return token  # Return unchanged if not numeric

    return '\n'.join(
        " ".join(evaluate_token(token) for token in line.split())
        for line in script.splitlines()
    )

def evaluate_lammps_arithmetic(script):
    """
    Replace LAMMPS-style $( … ) expressions that are *purely numeric*
    with their evaluated values.
    """
    # matches $(   3.14*1e2  )  but *not* $(x+1) or $(v_t+1)
    expr_pat = re.compile(
        r"""\$\(\s*([0-9+\-*/^(). eE]+?)\s*\)""",  # capture inner arithmetic
        flags=re.VERBOSE,
    )

    def repl(match: re.Match) -> str:
        expr = match.group(1).replace("^", "**")
        try:
            value = simpleeval.simple_eval(expr, names={"pi": math.pi}, functions={"sqrt": math.sqrt})
            if isinstance(value, float) and value.is_integer():
                value = int(value)
            return str(value)
        except Exception:
            # If evaluation fails (e.g., empty or invalid), leave it unchanged
            return match.group(0)

    return expr_pat.sub(repl, script)

def expand_loops(script):
    """Expands simple LAMMPS loops by evaluating 'if' conditions and unrolling iterations."""
    lines = script.splitlines()
    expanded_lines = []
    loop_label, loop_var_name, loop_count = None, None, None

    # **Step 1: Detect loop structure**
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('label '):
            loop_label = stripped.split()[1]
        elif stripped.startswith('variable ') and ' loop ' in stripped:
            parts = stripped.split()
            if len(parts) >= 4 and parts[2] == 'loop' and parts[3].isdigit():
                loop_var_name, loop_count = parts[1], int(parts[3])

    # **Step 2: Return early if no loop is detected**
    if not loop_var_name or not loop_count:
        return script  # No loop to expand

    # **Step 3: Expand loop iterations**
    for iteration in range(1, loop_count + 1):
        for line in lines:
            stripped = line.strip()

            # Skip loop control statements
            if stripped.startswith(f'variable {loop_var_name} loop') or \
               stripped.startswith(f'next {loop_var_name}') or \
               stripped.startswith(f'jump SELF {loop_label}'):
                continue

            # Process `if` conditions
            if stripped.startswith('if '):
                match = re.match(r'^if\s+"([^"]+)"\s+then\s+(.*)$', stripped)
                if match:
                    condition, then_part = match.groups()
                    condition_eval = condition.replace(f'${{{loop_var_name}}}', str(iteration))

                    if re.fullmatch(r'[\d\s<>=!]+', condition_eval):  # Ensure safe evaluation
                        try:
                            if simpleeval.simple_eval(condition_eval):
                                expanded_lines.extend(re.findall(r'"([^"]*)"', then_part))
                        except:
                            pass
                    continue  # Skip adding the original `if` line

            # Append regular lines
            expanded_lines.append(line)

    return '\n'.join(expanded_lines)

def sanitize(script):
    """Runs all sanitization steps in order, ensuring proper variable resolution and trailing newline."""
    script = remove_comments(script)
    script = remove_prints(script)
    script = merge_ampersand_lines(script)
    script = expand_loops(script)
    script = process_and_evaluate_variables(script)
    script = evaluate_expressions(script)
    script = evaluate_lammps_arithmetic(script)
    
    return script.rstrip() + '\n'

# Parsing

In [32]:
# Get the current working directory (useful in Jupyter/IPython)
current_dir = os.getcwd()
print(current_dir)
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
print(parent_dir)


# Add the parent directory to sys.path
sys.path.append(parent_dir)
##########

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample
/home/holbrooe/LAMMPS-AST


In [33]:
print(scripts_dir)
prompt_numbers = sorted(next(os.walk(scripts_dir))[1])
print(prompt_numbers)

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts
['prompt1', 'prompt2', 'prompt3']


In [34]:
#what we see

prompt_model_map1 = {}

for prompt in prompt_numbers:
    prompt_dir = os.path.join(scripts_dir, prompt)
    models = sorted(next(os.walk(prompt_dir))[1])  
    prompt_model_map1[prompt] = models

# Display detected structure
for prompt, models in prompt_model_map1.items():
    print(f"📂 {prompt}: {', '.join(models)}")

📂 prompt1: gpt-4o
📂 prompt2: gpt-4o
📂 prompt3: gpt-4o


In [35]:
# #what we do
# for selecting a custom set 
# prompt_model_map1 = {'prompt1': ['gpt-4.1', 'gpt-4o', 'gpt-o3','claude-4-opus'], 
#                      'prompt2': ['gpt-4.1', 'gpt-4o', 'gpt-o3','claude-4-opus'],
#                      'prompt3': ['gpt-4.1', 'gpt-4o', 'gpt-o3','claude-4-opus']}

In [36]:
# for selecting a custom set 

# print(prompt_model_map1)
# prompt_model_map2 = {'prompt1':['gpt-o3']}
# print(prompt_model_map2)

# Parse Stats

In [37]:
print(prompt_model_map1)

{'prompt1': ['gpt-4o'], 'prompt2': ['gpt-4o'], 'prompt3': ['gpt-4o']}


In [42]:
# dataframe initialization
df = pd.DataFrame([],columns=['prompt','model','trial'])

counter = 0
for prompt, models in prompt_model_map1.items():
    for model in models:
        for trial in np.arange(0,1,1): # change this line for more trials
        # for trial in np.arange(0,10,1): # change this line for more trials
            df.loc[counter,['prompt','model','trial']] = [prompt,model,trial] 

            counter += 1
            
display(df)

,prompt,model,trial
0,prompt1,gpt-4o,0
1,prompt2,gpt-4o,0
2,prompt3,gpt-4o,0


In [47]:
def parse_and_add_results(prompt_model_map, df, scripts_dir):
    """
    Given an existing df with columns ['prompt','model','trial',…],
    add a 'parsed' column ('Success'/'Fail') but don’t write any AST files.
    """
    df = df.copy()
    df['sanitized'] = 'n/a'
    df['parsed'] = 'Not Run'
    df['ast_path'] = None
    
    sanitized_scripts_dir = 'sanitized_scripts'
    os.makedirs(sanitized_scripts_dir, exist_ok=True)
    
    asts_dir = 'asts'
    os.makedirs(asts_dir, exist_ok=True)    
    
    df = df.set_index(['prompt','model','trial'])
    
    for prompt, models in prompt_model_map.items():
        for model in models:
            script_dir = os.path.join(scripts_dir, prompt, model)
            print(script_dir)
            for trial in range(10):
                print(prompt,model,trial)
                key = (prompt, model, trial)
                if key not in df.index:
                    continue

                script_file = f"{prompt}-{model}-T{trial}.in"
                script_path = os.path.join(script_dir, script_file)

                with open(script_path) as f:
                    src = f.read()
                sanitized = sanitize(src)
                print(sanitized)
                
                os.makedirs(os.path.join(sanitized_scripts_dir,prompt,model), exist_ok=True)
                sanitized_script_path = os.path.join(sanitized_scripts_dir,prompt,model,script_file)
                
                with open(sanitized_script_path, 'w') as output_file:
                    output_file.writelines(sanitized)

                ast_obj, errors = parse_to_AST(sanitized, lint=True, max_errors=10, verbose=False)

            
                if ast_obj is not None and len(errors) == 0:
                    result = True # "Success" originally used Success to differentiate the two 
                    flag = True
                    
                    ast_out_dir = os.path.join(asts_dir, prompt, model)
                    os.makedirs(ast_out_dir, exist_ok=True)
                    ast_path = os.path.join(ast_out_dir, script_file.replace(".in", ".ast.pkl"))
                    with open(ast_path, "wb") as pf:
                        pickle.dump(ast_obj, pf, protocol=pickle.HIGHEST_PROTOCOL)

                    df.loc[key, 'ast_path'] = ast_path
                    
                else:
                    # Store lint errors as JSON-safe dicts
                    result = json.dumps([e.__dict__ for e in errors])
                    first = errors[0] if errors else None
                    
                    if first is not None and first.text is not None:
                        tokens = first.text.split()
                        flag = not any(t.startswith(('v_', '$')) for t in tokens)
                        if not flag:
                            result = 'n/a'
                    else:
                        flag = False
                        result = 'n/a'
                    
                df.loc[key, 'sanitized'] = flag
                df.loc[key, 'parsed'] = result

    return df.reset_index()

In [48]:
new_df = parse_and_add_results(prompt_model_map1, df, scripts_dir)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(new_df)
    
new_df.to_pickle('parsing_df.pkl')

/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts/prompt1/gpt-4o
prompt1 gpt-4o 0
units metal
boundary p p p
atom_style atomic
lattice fcc 4.05
region box block 0 5 0 5 0 5
create_box 1 box
create_atoms 1 box
pair_style eam/alloy
pair_coeff * * ../../../potentials/prompt1.potential Al
timestep 0.001
thermo_style custom step temp press
thermo 100
fix 1 all npt temp 300 300 0.1 iso 1 1 1
run 500000

prompt1 gpt-4o 1
prompt1 gpt-4o 2
prompt1 gpt-4o 3
prompt1 gpt-4o 4
prompt1 gpt-4o 5
prompt1 gpt-4o 6
prompt1 gpt-4o 7
prompt1 gpt-4o 8
prompt1 gpt-4o 9
/home/holbrooe/LAMMPS-AST/EvaluationPipelineExample/generated_scripts/prompt2/gpt-4o
prompt2 gpt-4o 0
units metal
dimension 3
boundary p p p
atom_style atomic
lattice fcc 3.52
region box block 0 10 0 10 0 10
create_box 1 box
create_atoms 1 box
replicate 10 10 10
pair_style eam
pair_coeff * * ../../../potentials/prompt1.potential Ni
velocity all create 600 12345 mom yes rot yes dist gaussian
fix 1 all npt temp 300 2500 0.1 

,prompt,model,trial,sanitized,parsed,ast_path
0,prompt1,gpt-4o,0,True,True,asts/prompt1/gpt-4o/prompt1-gpt-4o-T0.ast.pkl
1,prompt2,gpt-4o,0,True,True,asts/prompt2/gpt-4o/prompt2-gpt-4o-T0.ast.pkl
2,prompt3,gpt-4o,0,True,True,asts/prompt3/gpt-4o/prompt3-gpt-4o-T0.ast.pkl
